## Vector Search

- https://github.com/Azure/azure-search-vector-samples/blob/main/demo-python/code/azure-search-vector-python-sample.ipynb
- https://learn.microsoft.com/en-us/azure/search/vector-search-how-to-query?tabs=query-2023-11-01,filter-2023-11-01

In [ ]:
%pip install azure-search-documents
%pip install azure-core
%pip install pydantic-settings
%pip install requests
%pip install openai
%pip install tiktoken
%pip install tenacitya
%pip install langchain
%pip install unstructured
%pip install "unstructured[pdf]"

In [ ]:
from pydantic_settings import BaseSettings

class Env(BaseSettings):
    SEARCH_API_KEY:str
    SEARCH_INDEX:str
    SEARCH_SERVICE_NAME:str
    OPENAI_API_KEY:str
    OPENAI_ENDPOINT:str
    OPENAI_API_VERSION:str = '2023-05-15'
    GPT_35_MODEL_NAME:str = "gpt-35-turbo"
    GPT_4_MODEL_NAME:str = "gpt-4"
    TEXT_EMBEDDING_MODEL_NAME:str="text-embedding-ada-002"
    class Config:
        env_file = "vector-search.env"
        
env = Env()

In [ ]:
from openai import AzureOpenAI
from tenacity import retry, wait_random_exponential, stop_after_attempt  

client = AzureOpenAI(api_key=env.OPENAI_API_KEY, api_version=env.OPENAI_API_VERSION, azure_endpoint=env.OPENAI_ENDPOINT)

@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# Function to generate embeddings for title and content fields, also used for query embeddings
def generate_embeddings(text, model=env.TEXT_EMBEDDING_MODEL_NAME):
    return client.embeddings.create(input = [text], model=model).data[0].embedding

## Create Index

In [12]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient  
from azure.search.documents.indexes.models import (
    SimpleField,
    SearchableField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchAlgorithmKind,
    HnswParameters,
    VectorSearchAlgorithmMetric,
    ExhaustiveKnnAlgorithmConfiguration,
    ExhaustiveKnnParameters,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
    SearchIndex,
    SemanticSearch,
    SearchField
)

# Create a search index
fields = [
    SimpleField(
        name="id",
        type=SearchFieldDataType.String,
        key=True,
        sortable=True,
        filterable=True,
    ),
    SearchableField(name="title", type=SearchFieldDataType.String, searchable=True, filterable=True),
    SearchableField(name="content", type=SearchFieldDataType.String, searchable=True, filterable=True),
    SearchField(
        name="title_embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="myHnswProfile",
    ),
    SearchField(
        name="content_embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=1536,
        vector_search_profile_name="myHnswProfile",
    ),
]

# Configure the vector search configuration
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw",
            kind=VectorSearchAlgorithmKind.HNSW,
            parameters=HnswParameters(
                m=4,
                ef_construction=400,
                ef_search=500,
                metric=VectorSearchAlgorithmMetric.COSINE,
            ),
        ),
        ExhaustiveKnnAlgorithmConfiguration(
            name="myExhaustiveKnn",
            kind=VectorSearchAlgorithmKind.EXHAUSTIVE_KNN,
            parameters=ExhaustiveKnnParameters(
                metric=VectorSearchAlgorithmMetric.COSINE
            ),
        ),
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw",
        ),
        VectorSearchProfile(
            name="myExhaustiveKnnProfile",
            algorithm_configuration_name="myExhaustiveKnn",
        ),
    ],
)

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name="title"),
        # keywords_fields=[SemanticField(field_name="category")],
        content_fields=[SemanticField(field_name="content")]
    )
)

# Create the semantic settings with the configuration
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create the search index with the semantic settings
index = SearchIndex(name=env.SEARCH_INDEX, fields=fields,
                    vector_search=vector_search, semantic_search=semantic_search)

index_client = SearchIndexClient(f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net", AzureKeyCredential(env.SEARCH_API_KEY))
result = index_client.create_or_update_index(index)
print(f' {result.name} created')

 kevin-test created


## Parse Documents with Langchain

In [13]:
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


# Read the PDF file
folder_path = "../../data"
loader = DirectoryLoader(folder_path, glob="*.pdf")
documents = loader.load()

print(f"Number of documents: {len(documents)}")

# # Split the document into paragraphs
splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
paragraphs = splitter.split_documents(documents)

print(f"Number of paragraphs: {len(paragraphs)}")

Ignoring (part of) ToUnicode map because the PDF data does not conform to the format. This could result in (cid) values in the output. The start object is not a byte.


Number of documents: 3
Number of paragraphs: 62


## Prepare Data for Index

In [14]:
from pathlib import Path
import os
import uuid
import json

embedded_paragraphs = []

for paragraph in paragraphs:
    filename = Path(paragraph.metadata.get('source','')).name
    title =  os.path.splitext(filename)[0]
    content = paragraph.page_content
    title_embedding = generate_embeddings(title)
    content_embedding = generate_embeddings(content)
    embedded_paragraphs.append({
        "id": uuid.uuid4().__str__(),
        "title": title,
        "content": content,
        "title_embedding": title_embedding,
        "content_embedding": content_embedding
    })

temp_json_path = "../../temp/docVectors.json"
with open(temp_json_path, "w", encoding="utf-8") as f:
    json.dump(embedded_paragraphs, f)

print("Done")

Done


## Insert Data to Index

### 少量資料

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import json

file = open(temp_json_path, 'r', encoding="utf-8")
upload_documents = json.load(file)

search_client = SearchClient(
    endpoint=f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    index_name=env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
)
result = search_client.upload_documents(upload_documents)
print(f"Uploaded {len(upload_documents)} documents") 

### 大量資料

In [15]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchIndexingBufferedSender
import json

file = open(temp_json_path, 'r', encoding="utf-8")
upload_documents = json.load(file)

with SearchIndexingBufferedSender(
    endpoint=f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    index_name=env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
) as batch_client:
    batch_client.upload_documents(documents=upload_documents)  
print(f"Uploaded {len(upload_documents)} documents in total")  

Uploaded 62 documents in total


## 向量搜尋
- 搜尋分數 `0.333 - 1.00` （Cosine），`0 ~ 1` 適用 Euclidean 和 DotProduct，分數**一律會隨著相似度變差而降低**

### 單一向量搜尋 (HNSW)


In [30]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery


query = "糖尿病注意事項"

search_client = SearchClient(
    f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
)
vector_query = VectorizedQuery(
    vector=generate_embeddings(query),
    k_nearest_neighbors=3,
    fields="content_embedding",
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "content"],
)

for result in results:
    print(f"{result['title']}")
    print(f"{result['content']}")
    print(f"Score: {result['@search.score']}\n")
    print("-" * 50)

糖尿病人高血糖注意事項
糖尿病人高血糖注意事項

壹、何謂急性高血糖？ 一、糖尿病酮酸中毒

多發生在第一型糖尿病，血糖值常高於 300mg/dL；且 伴隨著尿酮或血酮出現。
Score: 0.89865893

--------------------------------------------------
糖尿病人旅行時注意事項
英文病歷 兩份藥物 血糖機 預防低血糖

食物

點心 兩雙鞋

貳、旅行時，要注意什(cid:4662)事情？
Score: 0.88566947

--------------------------------------------------
糖尿病人高血糖注意事項
柒、複習一下 ( ) 問題 1：高血糖發生時會有多吃、多尿、口渴、體重減輕;等症狀。 ( ) 問題 2：西藥不宜吃太多所以如果吃感冒藥就把降血糖藥物停掉。 ( ) 問題
Score: 0.88379496

--------------------------------------------------


### Exhaustive KNN exact nearest neighbor search
- 適用於對於搜尋精準度要求高於搜尋效率的情境

In [22]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery


query = "糖尿病注意事項"

search_client = SearchClient(f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net", env.SEARCH_INDEX, credential=AzureKeyCredential(env.SEARCH_API_KEY))
vector_query = VectorizedQuery(vector=generate_embeddings(query), k_nearest_neighbors=3, fields="content_embedding", exhaustive=True)

results = search_client.search(  
    search_text=None,  
    vector_queries= [vector_query],
    select=["title", "content"],
)  

for result in results:
    print(f"{result['title']}")  
    print(f"{result['content']}")  
    print(f"Score: {result['@search.score']}\n")

糖尿病人高血糖注意事項
糖尿病人高血糖注意事項

壹、何謂急性高血糖？ 一、糖尿病酮酸中毒

多發生在第一型糖尿病，血糖值常高於 300mg/dL；且 伴隨著尿酮或血酮出現。
Score: 0.89865863

糖尿病人旅行時注意事項
英文病歷 兩份藥物 血糖機 預防低血糖

食物

點心 兩雙鞋

貳、旅行時，要注意什(cid:4662)事情？
Score: 0.88566947

糖尿病人高血糖注意事項
柒、複習一下 ( ) 問題 1：高血糖發生時會有多吃、多尿、口渴、體重減輕;等症狀。 ( ) 問題 2：西藥不宜吃太多所以如果吃感冒藥就把降血糖藥物停掉。 ( ) 問題
Score: 0.8837951

糖尿病人旅行時注意事項
四、每個國家的胰島素(cid:5146)度不一定相同，若(cid:4641)要在當地(cid:5564)

買，要特別注意。

攜帶糖尿病護照 隨身攜帶胰島素 每天監測血糖 注意
Score: 0.88373977

糖尿病人高血糖注意事項
參、什麼情況下會引起急性高血糖？

一、其他疾病如：感染、中風、胰臟炎、心肌梗塞等。 二、自行停止服用降血糖藥物。 三、藥物，如：類固醇、利尿劑…等。 四、情緒壓力。 五、不知道自己有糖尿病。
Score: 0.8806882

糖尿病人高血糖注意事項
亞東紀念醫院 祝您健康

1

感染發燒 中風

心肌梗塞

壓力

注射時間不當

肆、發生急性高血糖時，該怎麼辦？

一、意識清醒者多攝取水分，監測血糖並依醫囑服用血糖藥
Score: 0.8791728

糖尿病人胰島素藥物注意事項
糖尿病人胰島素藥物注意事項

壹、注射胰島素的原因 胰島素是人體胰臟所分泌的一種荷爾蒙，當糖尿病人 缺乏胰島素時，可直接補充胰島素治療糖尿病。

貳、胰島素藥物保存
Score: 0.87896997

糖尿病人高血糖注意事項
多吃

口渴

多尿

體重減輕

呼吸喘

意識改變

嘔吐

心跳快速

參、什麼情況下會引起急性高血糖？
Score: 0.8753362

糖尿病人高血糖注意事項
1.社團法人中華民國糖尿病衛教學會(2020)。糖尿病衛教核心教材。臺北 市：中華民國糖尿病衛教學會，p153-163。 2.衛生福利部國民健康署(2015)。糖尿病與我。臺北市：國民健康署。
Score: 0.8728

### 篩選 + 單一向量搜尋
透過 `filter` 限制搜尋結果，需確定欄位 `filterable = True`

In [31]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery, VectorFilterMode


query = "糖尿病注意事項"

search_client = SearchClient(
    f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
)
vector_query = VectorizedQuery(
    vector=generate_embeddings(query),
    k_nearest_neighbors=3,
    fields="content_embedding",
)

# 透過 filter 限制搜尋結果
results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["title", "content"],
    vector_filter_mode=VectorFilterMode.PRE_FILTER, # 預先過濾 filter 條件，再進行向量搜尋，也可以換成 POST_FILTER
    filter="title eq '糖尿病人旅行時注意事項'",
)

for result in results:
    print(f"{result['title']}")
    print(f"{result['content']}")
    print(f"Score: {result['@search.score']}\n")
    print("-" * 50)

糖尿病人旅行時注意事項
英文病歷 兩份藥物 血糖機 預防低血糖

食物

點心 兩雙鞋

貳、旅行時，要注意什(cid:4662)事情？
Score: 0.88566947

--------------------------------------------------
糖尿病人旅行時注意事項
四、每個國家的胰島素(cid:5146)度不一定相同，若(cid:4641)要在當地(cid:5564)

買，要特別注意。

攜帶糖尿病護照 隨身攜帶胰島素 每天監測血糖 注意
Score: 0.8837401

--------------------------------------------------
糖尿病人旅行時注意事項
四、預防低血(cid:5207)的食物，如：方(cid:5207)、果汁、汽水。
Score: 0.85987353

--------------------------------------------------


## Hybrid Search

- 混合式查詢，結果會依 RRF 排名

### RRF 演算法

- RRF 演算法是一種整合多種搜尋方法結果的排名方法。簡單來說，它先從多個平行執行的搜尋查詢中取得排名結果，然後為每個文件分配一個倒數排名分數，最後將這些分數結合起來創建一個新的排名。
- 在 Azure AI Search 當有兩個以上的查詢平行執行時，就會使用 RRF 演算法

具體步驟如下：

1. 從多個平行執行的搜尋查詢中獲取排名結果。
2. 為每個排名列表中的結果分配倒數排名分數。這是基於文檔在列表中的位置，分數按照公式 `1/(rank + k)` 計算，其中 rank 是文檔在列表中的位置，k 是一個經實驗證明效果較佳的常數，通常設置為 `60`。
3. 合併分數。對於每個文檔，將從每個搜索系統中獲得的倒數排名分數相加，形成合併分數。
4. 根據合併分數對文檔進行排名和排序，得到融合排名。
5. 只有標記為可搜尋（searchable）的索引字段或查詢中標記為 searchFields 的字段用於評分。
6. 搜尋結果僅返回標記為可檢索（retrievable）的字段或在查詢中指定的字段，同時返回它們的搜索分數。

情境：

1. 從系統 A 和系統 B 獲得以下的搜尋結果（只列出前三個結果作為範例）：
   系統 A: 蘋果果園, 紅蘋果, 蘋果醬
   系統 B: 紅蘋果, 蘋果汁, 蘋果果園
2. 接著，我們給每一個搜尋結果分配倒數排名分數。按照 1/(rank + 60)計算：

| 搜尋系統 | 搜尋結果 | 倒數排名分數 (1/(rank + 60)) |
| -------- | -------- | ---------------------------- |
| 系統 A   | 蘋果果園 | 0.016                        |
| 系統 A   | 紅蘋果   | 0.015                        |
| 系統 A   | 蘋果醬   | 0.014                        |
| 系統 B   | 紅蘋果   | 0.016                        |
| 系統 B   | 蘋果汁   | 0.015                        |
| 系統 B   | 蘋果果園 | 0.014                        |

3. 最後，根據這些合併的分數進行排序，最終給出的融合排名是

| 合併結果 | 總分數                        |
| -------- | ----------------------------- |
| 蘋果果園 | 0.030 (0.016 + 0.014 = 0.030) |
| 紅蘋果   | 0.031 (0.015 + 0.016 = 0.031) |
| 蘋果汁   | 0.015                         |
| 蘋果醬   | 0.014                         |


In [35]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.models import VectorizedQuery


query = "糖尿病注意事項"

search_client = SearchClient(
    f"https://{env.SEARCH_SERVICE_NAME}.search.windows.net",
    env.SEARCH_INDEX,
    credential=AzureKeyCredential(env.SEARCH_API_KEY),
)
vector_query = VectorizedQuery(
    vector=generate_embeddings(query),
    k_nearest_neighbors=3,
    fields="content_embedding",
)

# 同時透過 search_text 和 vector_query 進行搜尋
results = search_client.search(
    search_text=query,
    vector_queries=[vector_query],
    select=["title", "content"],
    top=3,
)

for result in results:
    print(f"{result['title']}")
    print(f"{result['content']}")
    print(f"Score: {result['@search.score']}\n")
    print("-" * 50)

糖尿病人高血糖注意事項
糖尿病人高血糖注意事項

壹、何謂急性高血糖？ 一、糖尿病酮酸中毒

多發生在第一型糖尿病，血糖值常高於 300mg/dL；且 伴隨著尿酮或血酮出現。
Score: 0.03333333507180214

--------------------------------------------------
糖尿病人旅行時注意事項
英文病歷 兩份藥物 血糖機 預防低血糖

食物

點心 兩雙鞋

貳、旅行時，要注意什(cid:4662)事情？
Score: 0.03226646035909653

--------------------------------------------------
糖尿病人高血糖注意事項
柒、複習一下 ( ) 問題 1：高血糖發生時會有多吃、多尿、口渴、體重減輕;等症狀。 ( ) 問題 2：西藥不宜吃太多所以如果吃感冒藥就把降血糖藥物停掉。 ( ) 問題
Score: 0.027893736958503723

--------------------------------------------------
